# data

> `TypedDecisions`: typed-decision cases from the Hub, JSON lines or pandas, split by case, turned into token rows by a `RowBuilder`, with the sampling strategies SetFit taught.

In [ ]:
#| default_exp data

In [ ]:
#| export
import hashlib, json, math, random, warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset, Features, Sequence, Value, load_dataset, get_dataset_config_names
from fastcore.basics import patch, store_attr

from sysone.core import *
from sysone.template import *

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail, test_ne, test_close
from transformers import AutoTokenizer
import datasets, transformers; datasets.disable_progress_bars(); transformers.utils.logging.disable_progress_bar()

One record format serves every application: a case with a `state`, its `questions` and, for training, the `gold` answers. `TypedDecisions` holds the cases of three splits, `train`, `valid` for metrics, and `calib`, a slice of train held out for temperature fitting. Splits are made by case, never by row, so the five questions of one state always land in the same split, and calibration never sees a row the model trained on (the Laya notebook's own correction). Rows are made later, once an encoder has been chosen, because a row is token ids and the tokenizer is the encoder's.

## Cases

Cases are stored as a `datasets.Dataset` with the state as text (JSON for structured states, as typed-decisions stores it) and the questions and gold as JSON strings. Heterogeneous question sets, five different questions per workflow, would otherwise become one wide struct with nulls.

The path from records to batches:

```{mermaid}
flowchart LR
    SRC["records: the Hub,<br/>JSON lines, pandas, converters"] --> C["cases<br/>(a datasets.Dataset)"]
    C --> SP{"split by case"}
    SP --> TR["train"]
    SP --> VA["valid"]
    TR --> CA["calib<br/>(held out of train)"]
    TR & VA & CA --> BI["bind(tokenizer, template)"]
    BI --> RO["rows(split): one row per question,<br/>datasets.map, cached by fingerprint"]
    RO --> CO["DecisionCollator:<br/>batches for the Trainer"]
```

In [ ]:
#| export
CASE_COLUMNS = ("id", "state", "questions", "gold")

def _enc_state(s):
    "A state as stored: JSON, so text states come back exactly as they went in"
    if isinstance(s, str) and s[:1] in "{[":               # typed-decisions' convention: structured states as JSON text
        try: s = json.loads(s)
        except json.JSONDecodeError: pass
    return json.dumps(s, ensure_ascii=False)
def _enc_json(v): return v if isinstance(v, str) else json.dumps(v, ensure_ascii=False)

def cases_from_records(records, prefix:str="case") -> Dataset:
    "A cases dataset from an iterable of records (dicts with `state`, `questions`, optional `gold` and `id`)"
    rows = []
    for i, r in enumerate(records):
        r = load_record(r, parse_state=False)
        extra = {k: (v if isinstance(v, (str, int, float, bool)) or v is None else json.dumps(v, ensure_ascii=False, default=str))
                 for k, v in r.items() if k not in CASE_COLUMNS}
        rows.append({"id": str(r.get("id", f"{prefix}_{i:06d}")), "state": _enc_state(r["state"]),
                     "questions": _enc_json(r["questions"]), "gold": _enc_json(r.get("gold") or {}), **extra})
    return Dataset.from_list(rows)

def as_cases(ds) -> Dataset:
    "A cases dataset from a `datasets.Dataset` with `state` and `questions` columns (strings or structures)"
    if not isinstance(ds, Dataset): return cases_from_records(ds)
    missing = {"state", "questions"} - set(ds.column_names)
    if missing: raise ValueError(f"cases need 'state' and 'questions' columns; missing {sorted(missing)}")
    cols = ["id", "state", "questions"] + (["gold"] if "gold" in ds.column_names else [])
    if all(c in ds.features and ds.features[c] == Value("string") for c in cols): return ds
    return cases_from_records(ds)

def case_record(cases:Dataset, i:int) -> dict:
    "Case i as a parsed record"
    return load_record(cases[i])

In [ ]:
recs = [json.loads(l) for l in Path("fixtures/typed_decisions_tiny.jsonl").read_text().splitlines()]
cs = cases_from_records(recs)
cs

Dataset({
    features: ['id', 'state', 'questions', 'gold', 'workflow'],
    num_rows: 20
})

In [ ]:
import pandas as pd

In [ ]:
test_eq(case_record(cs, 3)["questions"], recs[3]["questions"])
test_eq(cs[0]["workflow"], "agent_trace_observability")
test_eq(as_cases(cs), cs)
test_eq(cases_from_records([{"state": "text", "questions": {}}])[0]["id"], "case_000000")
test_eq(case_record(cases_from_records([{"state": '"quoted" text', "questions": {}}]), 0)["state"], '"quoted" text')
test_eq(len(cases_from_records(pd.DataFrame({"state": ["a"], "questions": [{}], "when": [pd.Timestamp("2026-01-01")]}).to_dict("records"))), 1)

### Splits by case

`split_cases` holds out a fraction (or a count) of cases with a seeded draw, keeping both parts in their original order. With `groups` it holds out whole groups instead, drawn in a seeded order until the fraction is reached, so no group has cases on both sides. `groups` names a column of the cases (a record's top-level field, such as a sequence cluster id) or is a function of the record:

In [ ]:
#| export
def case_groups(cases:Dataset, groups) -> list:
    "The group of each case: a column of the cases, or `groups(record)`"
    if callable(groups): return [groups(case_record(cases, i)) for i in range(len(cases))]
    if groups not in cases.column_names: raise ValueError(f"no column {groups!r} in the cases; name a top-level field of the records or pass a function")
    return list(cases[groups])

def split_cases(cases:Dataset, frac, seed:int=0, groups=None):
    "Two datasets: the cases kept and the `frac` (a fraction or a count) held out; with `groups`, whole groups are held out"
    n = len(cases)
    k = int(round(frac * n)) if isinstance(frac, float) and frac < 1 else int(frac)
    if k <= 0: return cases, None
    if k >= n: raise ValueError(f"cannot hold out {k} of {n} cases")
    if groups is None: held = set(random.Random(seed).sample(range(n), k))
    else:
        members = {}
        for i, g in enumerate(case_groups(cases, groups)): members.setdefault(g, []).append(i)
        order = sorted(members, key=str); random.Random(seed).shuffle(order)
        held = set()
        for g in order:
            if len(held) >= k: break
            held.update(members[g])
        if len(held) >= n: raise ValueError(f"holding out {k} of {n} cases by group leaves nothing to train on; there are {len(members)} groups")
    return cases.select([i for i in range(n) if i not in held]), cases.select(sorted(held))

In [ ]:
a, b = split_cases(cs, 0.25)
test_eq((len(a), len(b)), (15, 5))
test_eq(set(a["id"]) & set(b["id"]), set())
test_eq(split_cases(cs, 0.25)[1]["id"], b["id"])
test_eq(split_cases(cs, 0)[1], None)
ga, gb = split_cases(cs, 0.25, groups="workflow")          # the fixture's 20 cases come from 4 workflows, 5 each
test_eq(set(ga["workflow"]) & set(gb["workflow"]), set())
test_eq(len(gb), 5)
test_eq(split_cases(cs, 0.25, groups=lambda r: r["workflow"])[1]["id"], gb["id"])

Why cases and not rows: the five questions of a case share one state, so a split drawn over rows puts almost every held-out row's state in training too, and the validation score measures recall of states the model has read:

In [ ]:
row_ids = [(i, qid) for i in range(len(cs)) for qid in json.loads(cs[i]["questions"])]
held = set(random.Random(0).sample(range(len(row_ids)), len(row_ids) // 4))
seen = {row_ids[j][0] for j in range(len(row_ids)) if j not in held}
leaked = sum(row_ids[j][0] in seen for j in held) / len(held)
test_eq(leaked > 0.9, True)
f"a random quarter of the {len(row_ids)} rows held out: {leaked:.0%} of them ask about a state that training rows also contain; split by case: 0%"

'a random quarter of the 100 rows held out: 100% of them ask about a state that training rows also contain; split by case: 0%'

The same leak recurs one level up whenever cases come in families: proteins from one sequence cluster, photos from one shoot (the [images tutorial](tutorials/neomme_images.ipynb#photos-from-the-same-shoot) found 17 of 40 held-out hot dogs next to a training photo of the same scene), messages from one user. A split by case keeps each case whole, and a split by group keeps each family whole. The loaders take `groups=` and pass it to both held-out splits.

## TypedDecisions

In [ ]:
#| export
class TypedDecisions:
    "Typed-decision cases in train, valid and calib splits, and the token rows a `RowBuilder` makes from them"
    def __init__(self,
                 train, # training cases: a `datasets.Dataset` or a list of records
                 valid=None, # validation cases, for metrics
                 calib=None, # held-out training cases, for temperature fitting
                 template:RowTemplate|str|None=None, # row template; the encoder's default when None
                 tfms:list|None=None, # transforms, run on every record at training and inference
                 sampler=None, # a `RowSampler`; every record once when None
                 seed:int=0):
        self.cases = {"train": as_cases(train), "valid": as_cases(valid) if valid is not None else None,
                      "calib": as_cases(calib) if calib is not None else None}
        self.template, self.tfms, self.sampler, self.seed = template, list(tfms or []), sampler, seed
        self.builder, self._rows = None, {}

    def __repr__(self):
        n = ", ".join(f"{len(v)} {k}" for k, v in self.cases.items() if v is not None)
        t = self.builder.template if self.builder else self.template
        return f"TypedDecisions({n} cases · template {getattr(t, 'name', t) or 'default'} · {len(self.tfms)} tfms)"

### Loaders

`from_records`, `from_json`, `from_pandas` and `from_hub` all make the same object. `valid` takes a split name (a Hub split, or `"test"`) or a fraction of train; `calib` holds out a fraction of the remaining train cases, 10% by default. `from_hub` picks the `all` config when a dataset has several and no config is named, as typed-decisions does.

In [ ]:
#| export
def _holdouts(train, valid, calib, seed, groups=None):
    if isinstance(valid, (int, float)) and not isinstance(valid, bool): train, valid = split_cases(train, valid, seed, groups)
    if isinstance(calib, (int, float)) and not isinstance(calib, bool): train, calib = split_cases(train, calib, seed + 1, groups)
    return train, valid, calib

@patch(cls_method=True)
def from_records(cls:TypedDecisions, records, valid=None, calib=0.1, seed:int=0, groups=None, **kw):
    "Cases from a list of records; `groups` (a field or a function of the record) keeps each group on one side of every split"
    train, valid, calib = _holdouts(cases_from_records(records), valid, calib, seed, groups)
    return cls(train, valid, calib, seed=seed, **kw)

def read_jsonl(path) -> list:
    "Records from a JSON-lines file (or a JSON list)"
    text = Path(path).read_text()
    if text.lstrip().startswith("["): return json.loads(text)
    return [json.loads(l) for l in text.splitlines() if l.strip()]

@patch(cls_method=True)
def from_json(cls:TypedDecisions, path, valid=None, calib=0.1, seed:int=0, groups=None, **kw):
    "Cases from a JSON-lines file, one record per line; `valid` may be a second file"
    v = cases_from_records(read_jsonl(valid)) if isinstance(valid, (str, Path)) else valid
    return cls.from_records(read_jsonl(path), valid=v, calib=calib, seed=seed, groups=groups, **kw)

@patch(cls_method=True)
def from_pandas(cls:TypedDecisions, df, valid=None, calib=0.1, seed:int=0, groups=None, **kw):
    "Cases from a DataFrame with `state` and `questions` (and `gold`) columns"
    return cls.from_records(df.to_dict("records"), valid=valid, calib=calib, seed=seed, groups=groups, **kw)

def _pick_config(path, config):
    if config is not None: return config
    try: names = get_dataset_config_names(path)
    except Exception: return None
    if len(names) <= 1: return names[0] if names else None
    for c in ("all", "default"):
        if c in names: return c
    raise ValueError(f"{path} has several configs; name one of {names}")

@patch(cls_method=True)
def from_hub(cls:TypedDecisions, path:str, config:str|None=None, split:str="train", valid=None, calib=0.1,
             revision:str|None=None, seed:int=0, groups=None, **kw):
    "Cases from a Hub dataset in the typed-decisions format; `valid` is a split name or a fraction of `split`"
    config = _pick_config(path, config)
    train = as_cases(load_dataset(path, config, split=split, revision=revision))
    if isinstance(valid, str): valid = as_cases(load_dataset(path, config, split=valid, revision=revision))
    train, valid, calib = _holdouts(train, valid, calib, seed, groups)
    return cls(train, valid, calib, seed=seed, **kw)

In [ ]:
data = TypedDecisions.from_json("fixtures/typed_decisions_tiny.jsonl", valid=0.2, calib=0.1)
data

TypedDecisions(14 train, 4 valid, 2 calib cases · template default · 0 tfms)

In [ ]:
test_eq({k: len(v) for k, v in data.cases.items()}, {"train": 14, "valid": 4, "calib": 2})
test_eq(TypedDecisions.from_records(recs, calib=0).cases["calib"], None)
gd = TypedDecisions.from_records(recs, valid=0.25, calib=0.2, groups="workflow")
test_eq(set(gd.cases["train"]["workflow"]) & (set(gd.cases["valid"]["workflow"]) | set(gd.cases["calib"]["workflow"])), set())

In [ ]:
#| eval: false
td = TypedDecisions.from_hub("LocalLLaMA/typed-decisions", valid="test")
td

```
TypedDecisions(1080 train, 400 valid, 120 calib cases · template default · 0 tfms)
```

## Rows

`bind` gives the data a tokenizer (or a processor), a template and any side streams, and `rows(split)` turns the split's cases into rows with `datasets.map`: one row per question, `input_ids`, `markers`, `qtype`, the gold `target` and `label`, whether the gold `answerable` the question, the option `spans`, each side stream's ids and mask, the length `n_tokens` for length-grouped batching, and the case and question ids. A row without gold is dropped, since it has nothing to teach; an unanswerable one is kept, with a zero target that the option loss skips, because it teaches the act head to escalate. The map's fingerprint is the cases' fingerprint plus the builder's (template, transforms and tokenizer), so a Hub dataset re-uses its cached rows on the next run with the same encoder, and any change rebuilds them.

In [ ]:
#| export
ROW_FEATURES = Features({"input_ids": Sequence(Value("int32")), "markers": Sequence(Value("int32")), "qtype": Value("int8"),
                         "target": Sequence(Value("float32")), "label": Value("int32"), "answerable": Value("int8"),
                         "n_tokens": Value("int32"), "case": Value("string"), "qid": Value("string"), "case_idx": Value("int64"),
                         "variant": Value("int32")})

def row_features(builder:RowBuilder|None=None) -> Features:
    "The columns of a split's rows: `ROW_FEATURES`, the option spans, and each side stream's ids and mask"
    f = dict(ROW_FEATURES) | {"spans": Sequence(Sequence(Value("int32")))}
    for s in (builder.streams.values() if builder is not None else []): f |= {s.columns[0]: Sequence(Value("int32")), s.columns[1]: Sequence(Value("int8"))}
    return Features(f)

@patch
def bind(self:TypedDecisions, tokenizer, template:RowTemplate|str|None=None, streams=None):
    "Build rows with `tokenizer` (or a processor), `template` (else the data's, else the `auto` preset) and side `streams`"
    template = template if template is not None else self.template if self.template is not None else "auto"
    self.builder = RowBuilder(tokenizer, template, self.tfms, streams=streams)
    self._rows = {}
    return self

def _builder_key(b:RowBuilder) -> str:
    "What the rows depend on: template, transforms, tokenizer (and processor), side streams, and the library's version"
    import sysone
    tok = b.tok
    streams = json.dumps([s.to_json() | {"tok": getattr(s.tokenizer, "name_or_path", ""), "n": len(s.tokenizer)} for s in b.streams.values()])
    return f"{b.template.hash}:{hashlib.sha1(json.dumps(tfms_to_json(b.tfms), sort_keys=True).encode()).hexdigest()[:8]}:" \
           f"{getattr(tok, 'name_or_path', '')}:{len(tok)}:{type(b.processor).__name__}:{sysone.__version__}:" \
           f"{hashlib.sha1(streams.encode()).hexdigest()[:8]}"

In [ ]:
#| export
def _rows_fn(builder, train, sampler):
    cols, stream_cols = row_features(builder), [c for s in builder.streams.values() for c in s.columns]
    def fn(batch, idx):
        out = {k: [] for k in cols}
        for i in range(len(batch["id"])):
            rec = load_record({k: batch[k][i] for k in batch})
            for v, r in enumerate(sampler.expand(rec, builder) if sampler is not None and train else [rec]):
                for row in builder.build_record(r, train=train, skip_errors=True):
                    if row.target is None and row.answerable != 0: continue     # no gold at all: nothing to learn from
                    d = row.to_dict()
                    if d["target"] is None: d["target"] = [0.0] * row.k                 # unanswerable: the option loss skips it
                    for k in ("input_ids", "markers", "qtype", "target", "label", "answerable", "n_tokens"): out[k].append(d[k])
                    out["spans"].append(d.get("spans") or [])
                    for c in stream_cols: out[c].append(d[c])
                    out["case"].append(str(row.meta.get("case"))); out["qid"].append(row.meta["qid"])
                    out["case_idx"].append(idx[i]); out["variant"].append(v)
        return out
    return fn

@patch
def _check_bound(self:TypedDecisions):
    if self.builder is None: raise ValueError("no tokenizer yet: call data.bind(tokenizer) or make a learner first")

@patch
def rows(self:TypedDecisions, split:str="train") -> Dataset:
    "The rows of a split (built once per binding)"
    self._check_bound()
    if split in self._rows: return self._rows[split]
    cases = self.cases[split]
    if cases is None: return None
    train = split == "train"
    fp = hashlib.sha1(f"{cases._fingerprint}:{_builder_key(self.builder)}:{train}:{self.sampler!r}".encode()).hexdigest()
    rows = cases.map(_rows_fn(self.builder, train, self.sampler), batched=True, batch_size=64, with_indices=True,
                     remove_columns=cases.column_names, features=row_features(self.builder), new_fingerprint=fp,
                     desc=f"rows ({split})")
    if train and self.sampler is not None: rows = self.sampler.resample(rows)
    s = self.builder.stats
    if s["rows"] and s["state_truncated"] / s["rows"] > 0.2:
        warnings.warn(f"{s['state_truncated']} of {s['rows']} rows had their state truncated to fit max_len="
                      f"{self.builder.template.max_len}; consider a Truncate transform or a longer budget")
    if s["dropped"]: warnings.warn(f"{s['dropped']} rows dropped: their options do not fit the template's budgets")
    self._rows[split] = rows
    return rows

In [ ]:
tok = AutoTokenizer.from_pretrained("fixtures/tiny-modernbert")
data.bind(tok, "laya")
tr = data.rows("train")
tr

Dataset({
    features: ['input_ids', 'markers', 'qtype', 'target', 'label', 'answerable', 'n_tokens', 'case', 'qid', 'case_idx', 'variant', 'spans'],
    num_rows: 70
})

In [ ]:
test_eq(len(tr), 14 * 5)
test_eq(set(tr["qtype"]), {0, 1, 2})
r = Row.from_dict(tr[0])
test_eq([r.ids[m] for m in r.markers], [tok.mask_token_id] * r.k)
test_eq([r.ids[s - 1] for s, e in r.spans], [tok.mask_token_id] * r.k)
test_eq(len(data.rows("valid")), 20)
test_eq(set(tr["answerable"]), {1})

A side stream's inputs are columns of the rows, and an unanswerable case stays in: a record marked `{"answerable": false}` gives a row with a zero target and `answerable` 0, while a case without gold gives no row. The fixture's text tokenizer stands in for a stream's own tokenizer:

In [ ]:
srecs = [dict(r, state={"text": r["state"] if isinstance(r["state"], str) else json.dumps(r["state"]), "notes": f"note {i}"}) for i, r in enumerate(recs[:4])]
q0 = next(iter(srecs[0]["questions"]))
srecs[1] = dict(srecs[1], gold={**srecs[1]["gold"], next(iter(srecs[1]["questions"])): {"answerable": False}})
sd = TypedDecisions.from_records(srecs, calib=0, tfms=[Stream("notes")]).bind(tok, "laya", streams=[TokenStream("notes", tok, max_len=16)])
srows = sd.rows("train")
test_eq({"notes_input_ids", "notes_attention_mask", "spans", "answerable"} <= set(srows.column_names), True)
test_eq(sorted(set(srows["answerable"])), [0, 1])
un = srows.filter(lambda r: r["answerable"] == 0)[0]
test_eq((sum(un["target"]), un["label"]), (0.0, -1))
test_eq(collate([srows[0], srows[1]])["notes_input_ids"].shape[0], 2)

### show_batch

`show_batch` prints rows with their anchors visible. Before a tokenizer is bound it renders the template as text instead, which is enough to see the layout:

In [ ]:
#| export
def _render_text(template:RowTemplate, record:dict, qid:str) -> str:
    "The row as text, before tokenization"
    q = as_questions(record["questions"])[qid]
    opts = template.option_sep.join(template.option.format(mask=template.mask, text=o) for o in q.options)
    vals = {r: (getattr(template, r) or "") for r in SPECIALS} | {"type": q.type, "instructions": q.instructions,
            "options": opts, "state": render_state(record["state"]), "image": template.image if record.get("images") else ""}
    return template.row.format(**vals)

@patch
def show_batch(self:TypedDecisions, n:int=2, split:str="train", max_chars:int=300):
    "Print the first `n` rows of a split"
    cases = self.cases[split]
    if self.builder is None:
        t = self.template if isinstance(self.template, RowTemplate) else \
            RowTemplate.preset(self.template if self.template not in (None, "auto") else "laya")
        shown = 0
        for i in range(len(cases)):
            rec = apply_tfms(list(t.tfms) + self.tfms, case_record(cases, i))
            for qid in rec["questions"]:
                s = _render_text(t, rec, qid)
                print((s if len(s) <= max_chars else s[:max_chars * 2 // 3] + " … " + s[-max_chars // 3:]) + "\n")
                shown += 1
                if shown >= n: return
        return
    rows = self.rows(split)
    for i in range(min(n, len(rows))): print(show_row(Row.from_dict(rows[i]), self.builder.tok, max_chars), end="\n\n")

In [ ]:
TypedDecisions.from_records(recs[:1], calib=0).show_batch(1)

[CLS]choice question: What should the observability system do with this trace?[SEP][MASK] continue: Let the agent proceed without interruption.[MASK] human_review: Queue this trace for a human to revi … _violations": 0, "duration_s": 128.5, "irreversible_actions": 0, "steps": 7, "tool_errors": 1}}[SEP]



In [ ]:
data.show_batch(2)

[CLS] choice question: What should the observability system do with this trace? [SEP] [MASK] continue: Let the agent proceed without interruption. [MASK] human_review: Queue this trace for a human to  … violations": 0, "duration_s": 128.5, "irreversible_actions": 0, "steps": 7, "tool_errors": 1}} [SEP]
markers [20, 31, 45, 60] · qtype choice · 163 tokens

[CLS] noul question: This trace requires human review. [SEP] [MASK] false: No human attention is warranted. [MASK] true: A human should inspect this run. [SEP] {"agent": {"autonomy": "checkpointed", " … violations": 0, "duration_s": 128.5, "irreversible_actions": 0, "steps": 7, "tool_errors": 1}} [SEP]
markers [15, 24] · qtype noul · 129 tokens



### Collation

`DecisionCollator` pads a list of dataset rows into a `Batch` for `transformers.Trainer`; it is `core.collate` with the tokenizer's pad id, dropping the bookkeeping columns:

In [ ]:
#| export
class DecisionCollator:
    "Collate dataset rows into a padded `Batch`"
    def __init__(self, pad_id:int=0): self.pad_id = pad_id
    def __call__(self, rows):
        b = collate(rows, self.pad_id)
        return b
    def __repr__(self): return f"DecisionCollator(pad_id={self.pad_id})"

@patch(as_prop=True)
def collator(self:TypedDecisions) -> DecisionCollator:
    "The collator for this data's tokenizer"
    self._check_bound()
    return DecisionCollator(self.builder.tok.pad_token_id or 0)

In [ ]:
b = data.collator([tr[i] for i in range(6)])
test_eq(b["input_ids"].shape[0], 6)
test_eq(b["target"].shape, b["marker_mask"].shape)
{k: tuple(v.shape) for k, v in b.items()}

{'input_ids': (6, 178),
 'attention_mask': (6, 178),
 'marker_pos': (6, 4),
 'marker_mask': (6, 4),
 'qtype': (6,),
 'target': (6, 4),
 'label': (6,),
 'spans': (6, 4, 2),
 'answerable': (6,)}

### Lazy rows

Two cases cannot be pre-built into a static dataset: augmentation that must differ every epoch (`Shuffle`, a resampled negative set), and images, whose pixel values are megabytes per row. For both, `lazy_rows` returns a dataset that builds each row when it is read, from its case, with the current epoch's randomness; its `lengths` come from the static build so length-grouped sampling still works.

In [ ]:
#| export
class LazyRows(torch.utils.data.Dataset):
    "Rows built when read, from their case: for per-epoch augmentation and for image rows"
    def __init__(self, data:TypedDecisions, split:str="train"):
        self.data, self.split = data, split
        static = data.rows(split)
        self.index = list(zip(static["case_idx"], static["variant"], static["qid"], static["case"]))
        self.lengths = static["n_tokens"]
    def __len__(self): return len(self.index)
    def __getitem__(self, i):
        ci, v, qid, case = self.index[i]
        train = self.split == "train"
        rec = case_record(self.data.cases[self.split], ci)
        if train and self.data.sampler is not None: rec = self.data.sampler.expand(rec, self.data.builder)[v]
        rec["questions"] = {qid: rec["questions"][qid]}
        rec["gold"] = {qid: (rec.get("gold") or {}).get(qid)}
        row = self.data.builder.build_record(rec, train=train)[0].to_dict()
        return row | {"case": case, "qid": qid, "case_idx": ci, "variant": v}      # which case and question, as the static rows say

@patch
def lazy_rows(self:TypedDecisions, split:str="train") -> LazyRows:
    "Rows of a split, built on read"
    self._check_bound()
    return LazyRows(self, split)

@patch(as_prop=True)
def needs_lazy_rows(self:TypedDecisions) -> bool:
    "Whether training rows must be rebuilt on read (train-only transforms, or a processor with images)"
    self._check_bound()
    return any(t.train_only for t in self.builder.tfms) or self.builder.processor is not None

In [ ]:
sh = TypedDecisions.from_records(recs, calib=0, tfms=[Shuffle(seed=0)]).bind(tok, "laya")
test_eq(sh.needs_lazy_rows, True)
lz = sh.lazy_rows()
test_eq(len(lz), 100)
e0 = [lz[i]["input_ids"] for i in range(10)]
sh.tfms[0].set_epoch(1)
e1 = [lz[i]["input_ids"] for i in range(10)]
test_ne(e0, e1)                                   # new option orders in a new epoch
test_eq([sorted(x) for x in e0], [sorted(x) for x in e1])
test_close(sorted(lz[0]["target"]), sorted(sh.rows()[0]["target"]), eps=1e-6)
ids = ("case", "qid", "case_idx", "variant")
test_eq([{k: lz[i][k] for k in ids} for i in range(5)], [{k: sh.rows()[i][k] for k in ids} for i in range(5)])   # a row built on read knows its case

Rows are found again by their case's position, not its id, so cases that share an id (two converted datasets concatenated, say) still come back as themselves:

In [ ]:
twins = [dict(recs[0], id="same"), dict(recs[5], id="same")]
tw = TypedDecisions.from_records(twins, calib=0, tfms=[Shuffle()]).bind(tok, "laya")
for i, ci in ((0, 0), (5, 1)):
    ref = tw.builder.build_record(case_record(tw.cases["train"], ci), train=True)[0]
    test_eq((tw.lazy_rows()[i]["input_ids"], tw.lazy_rows()[i]["label"]), (ref.ids, ref.label))

## Sampling

SetFit's contrastive phase does not transfer: the softmax across a row's options already is a contrastive objective, the gold option against the distractors the task designer wrote. Its sampling strategies do, adapted to rows. `RowSampler` has four:

- `all`: every record once, every option in its row (the default);
- `oversample`: repeat rows whose gold label is rare for their question, cycling through them (SetFit's wrap-around oversampling) until every label of a question is as frequent as its commonest;
- `negatives=k`: for choice questions with more than k options, keep the gold option and k − 1 distractors drawn at random, renormalising the soft target over the kept ones (GLiNER's negative sampling);
- `hard`: like `negatives`, but the distractors are the options most similar to the gold one under a sentence-transformer, so the row's negatives are the confusable ones (needs `sentence-transformers`).

`negatives` and `hard` draw per record; `variants=n` draws n versions of every record, which is how they combine with a static feature cache.

In [ ]:
#| export
class RowSampler:
    "Which rows a training epoch sees: `all`, `oversample`, or `negatives=k` / `hard` distractor sampling"
    def __init__(self, strategy:str="all", negatives:int|None=None, variants:int=1, model:str="BAAI/bge-small-en-v1.5", seed:int=0):
        if strategy not in ("all", "oversample", "negatives", "hard"): raise ValueError(f"unknown strategy {strategy!r}")
        if strategy in ("negatives", "hard") and not negatives: raise ValueError(f"strategy {strategy!r} needs negatives=k")
        store_attr(); self._embedder = None
    def __repr__(self): return f"RowSampler({self.strategy}, negatives={self.negatives}, variants={self.variants}, seed={self.seed})"

In [ ]:
#| export
def _gold_key(q:Question, gold):
    t, label = q.target(gold)
    return None if t is None else q.keys[label]

@patch
def _similar(self:RowSampler, texts:list, anchor:int) -> list:
    "Indices of `texts` by decreasing similarity to texts[anchor]"
    if self._embedder is None:
        from sentence_transformers import SentenceTransformer
        self._embedder = SentenceTransformer(self.model)
    e = self._embedder.encode(texts, normalize_embeddings=True)
    sims = e @ e[anchor]
    return [int(i) for i in np.argsort(-sims) if i != anchor]

@patch
def _sample_question(self:RowSampler, q:Question, gold, rng):
    "A definition of `q` with k options: the gold one and k − 1 distractors"
    k = self.negatives
    if not isinstance(q, Choice) or q.k <= k: return q.to_dict(), gold
    keys = list(q.criteria)
    g = _gold_key(q, gold)
    gi = keys.index(g) if g in keys else rng.randrange(len(keys))
    if self.strategy == "hard": pool = self._similar(q.options, gi)[:max(k - 1, 2 * (k - 1))]
    else: pool = [i for i in range(len(keys)) if i != gi]
    keep = sorted([gi] + rng.sample(pool, k - 1))
    crit = {keys[i]: q.criteria[keys[i]] for i in keep}
    if isinstance(gold, dict) and isinstance(gold.get("probabilities"), dict):
        gp = {str(k): v for k, v in gold["probabilities"].items()}
        p = {kk: gp.get(str(kk), 0.0) for kk in crit}
        s = sum(p.values()) or 1.0
        gold = {**gold, "probabilities": {kk: v / s for kk, v in p.items()}}
    return {**q.to_dict(), "criteria": crit}, gold

@patch
def expand(self:RowSampler, record:dict, builder=None) -> list:
    "The records one training record becomes (`variants` draws for the negative strategies)"
    if self.strategy not in ("negatives", "hard"): return [record]
    record = load_record(record, parse_state=False)
    out = []
    for v in range(self.variants):
        rng = random.Random(f"{self.seed}:{v}:{record.get('id')}")
        qs, gold = {}, dict(record.get("gold") or {})
        for qid, q in as_questions(record["questions"]).items():
            qs[qid], g = self._sample_question(q, gold.get(qid), rng)
            if qid in gold: gold[qid] = g
        out.append({**record, "questions": qs, "gold": gold, "id": f"{record.get('id')}" + (f"#{v}" if v else "")})
    return out

@patch
def resample(self:RowSampler, rows:Dataset) -> Dataset:
    "Oversample rare labels per question, cycling through their rows"
    if self.strategy != "oversample": return rows
    groups = defaultdict(lambda: defaultdict(list))
    for i, (qid, label) in enumerate(zip(rows["qid"], rows["label"])): groups[qid][label].append(i)
    idx = []
    for qid, by_label in groups.items():
        top = max(len(v) for v in by_label.values())
        for label, members in by_label.items(): idx += [members[j % len(members)] for j in range(top)]
    return rows.select(sorted(idx))

In [ ]:
many = {"state": "The printer on floor 3 is jammed again.",
        "questions": {"dept": {"type": "choice", "instructions": "Which team?",
                               "criteria": {f"team {i}": f"handles area {i}" for i in range(12)}}},
        "gold": {"dept": {"label": "team 7", "probabilities": {"team 7": 0.8, "team 2": 0.2}}}}
neg = RowSampler("negatives", negatives=4, variants=2)
vs = neg.expand(many)
test_eq(len(vs), 2)
test_eq(len(vs[0]["questions"]["dept"]["criteria"]), 4)
assert "team 7" in vs[0]["questions"]["dept"]["criteria"] and "team 7" in vs[1]["questions"]["dept"]["criteria"]
test_ne(list(vs[0]["questions"]["dept"]["criteria"]), list(vs[1]["questions"]["dept"]["criteria"]))
test_eq(sum(vs[0]["gold"]["dept"]["probabilities"].values()), 1.0)

In [ ]:
ov = TypedDecisions.from_records(recs, calib=0, sampler=RowSampler("oversample")).bind(tok, "laya")
counts = Counter((q, l) for q, l in zip(ov.rows()["qid"], ov.rows()["label"]))
per_q = defaultdict(set)
for (q, l), c in counts.items(): per_q[q].add(c)
test_eq(all(len(v) == 1 for v in per_q.values()), True)   # every label of a question equally frequent

In [ ]:
ng = TypedDecisions.from_records([many] * 3, calib=0, sampler=RowSampler("negatives", negatives=3, variants=2)).bind(tok, "laya")
test_eq(len(ng.rows()), 6)
test_eq(set(len(m) for m in ng.rows()["markers"]), {3})

Rows built on read follow the sampler too, variant by variant, and integer labels keep their soft gold through the sampling:

In [ ]:
ngl = TypedDecisions.from_records([many] * 3, calib=0, tfms=[Shuffle()], sampler=RowSampler("negatives", negatives=3, variants=2)).bind(tok, "laya")
test_eq([len(ngl.lazy_rows()[i]["markers"]) for i in range(6)], [3] * 6)
ints = {"state": "x", "questions": {"q": {"type": "choice", "instructions": "?", "criteria": list(range(10))}},
        "gold": {"q": {"label": 7, "probabilities": {"7": 0.7, "3": 0.3}}}}
kept = RowSampler("negatives", negatives=4).expand(ints)[0]
test_eq(Question.from_dict(kept["questions"]["q"]).target(kept["gold"]["q"])[0] != [0.25] * 4, True)
test_close(max(kept["gold"]["q"]["probabilities"].values()), 0.7 if 3 in kept["questions"]["q"]["criteria"] else 1.0)

## Shortlists

When a question has more options than a row can hold well (Laya degrades above about twenty), a bi-encoder can shortlist them first: `Shortlist` embeds the option texts and the state, keeps the top k options by cosine similarity, and the decision is made among those. Dropped options are recorded in the record, so the `Decider` reports them with probability 0 and a flag; the same transform builds the training rows, so training and inference match. It is the rerank-cookbook pattern, and the answer to wide label sets that does not touch the budgets. It needs `sentence-transformers`.

In [ ]:
#| export
@transform("Shortlist")
class Shortlist(Transform):
    "Keep the k options of each choice question most similar to the state under a sentence-transformer"
    def __init__(self, model:str="BAAI/bge-small-en-v1.5", k:int=12, min_options:int|None=None):
        store_attr(); self._enc = None
    def _embed(self, texts):
        if self._enc is None:
            from sentence_transformers import SentenceTransformer
            self._enc = SentenceTransformer(self.model)
        return self._enc.encode(texts, normalize_embeddings=True)
    def __call__(self, record):
        qs, dropped = {}, dict(record.get("dropped") or {})
        state = render_state(record["state"])
        for qid, qd in record["questions"].items():
            q = Question.from_dict(qd, name=qid)
            if not isinstance(q, Choice) or q.k <= max(self.k, self.min_options or 0): qs[qid] = qd; continue
            e = self._embed([state] + q.options)
            order = np.argsort(-(e[1:] @ e[0]))[:self.k]
            keys = list(q.criteria)
            keep = [keys[i] for i in sorted(order)]
            qs[qid] = {**q.to_dict(), "criteria": {kk: q.criteria[kk] for kk in keep}}
            dropped[qid] = [kk for kk in keys if kk not in keep]
        return {**record, "questions": qs, "dropped": dropped}

In [ ]:
#| eval: false
sl = Shortlist(k=3)
out = sl(many)
out["questions"]["dept"]["criteria"], out["dropped"]["dept"]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()